
COMPUTER VISION PROJECT - Assistance for Visually Impaired People
Authors: Victor Micha, Nawel Zait
Date: December 2025

This system combines 4 AI modules to help visually impaired users navigate:
- Module 1: Object Detection (YOLO) - Detects obstacles
- Module 2: OCR (EasyOCR) - Reads text (signs, street names)
- Module 3: Scene Description (BLIP) - Describes environment
- Module 4: Object Classification (ResNet) - Identifies main objects

Features:
- Real-time video processing
- Intelligent voice alerts (TTS)
- Distance estimation
- Alert prioritization (danger > warning > info)

Requirements:
- Python 3.8+
- CUDA (optional, for GPU acceleration)
- Internet connection (for gTTS)


# Setup

## Librairies

In [ ]:
# Détection d'objets
!pip install ultralytics  # YOLO v8
!pip install torch torchvision

# OCR
!pip install easyocr pytesseract

# Vision-Language
!pip install transformers pillow

# Utilitaires
!pip install opencv-python-headless numpy matplotlib

In [ ]:
!pip install ultralytics opencv-python-headless pillow numpy matplotlib
!pip install gtts playsound  # Pour la synthèse vocale


In [ ]:
!pip install easyocr pytesseract
!pip install opencv-python-headless pillow numpy matplotlib
!apt-get install -y tesseract-ocr tesseract-ocr-fra  # Support français

In [16]:
# EasyOCR - Meilleur pour texte dans images naturelles (panneaux, enseignes)
reader_easyocr = easyocr.Reader(['fr', 'en'], gpu=True)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
!pip install -q transformers accelerate pillow torch torchvision
!pip install -q salesforce-lavis  # Pour BLIP-2


In [28]:
processor_blip = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
#NB : possible d'utiliser un modèle plus récent, type BCLIP-2, mais sera plus lent donc je préfère commencer par lui.

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [53]:
# 1. Installation de gTTS (correct)
!pip install gtts

In [66]:
# Installation du module speech recognition
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 33.7 MB/s eta 0:00:00


In [50]:
!pip install pyttsx3

## Imports

In [3]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch
from pathlib import Path
from collections import Counter

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

PyTorch version: 2.9.0+cu126
CUDA disponible: True


In [5]:
# Chargement du modèle YOLOv8 pré-entraîné sur COCO
# Options: yolov8n (nano - rapide), yolov8s (small), yolov8m (medium), yolov8l (large)
model = YOLO('yolov8n.pt')
print(f"Classes disponibles: {len(model.names)} catégories")

Classes disponibles: 80 catégories


In [15]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import easyocr
import pytesseract
from collections import defaultdict
import re

In [26]:
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import cv2
import numpy as np
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé: {device}")

Device utilisé: cuda


# Code

In [109]:
"""
COMPUTER VISION PROJECT - Assistance for Visually Impaired People
Authors: Victor Micha, Nawel Zait
Date: December 2024

System combining 4 AI modules:
- Module 1: Object Detection (YOLO)
- Module 2: OCR (EasyOCR)
- Module 3: Scene Description (BLIP)
- Module 4: Object Classification (ResNet)
"""

# ============================================
# INSTALLATIONS (run once)
# ============================================
# !pip install ultralytics opencv-python-headless pillow numpy matplotlib
# !pip install easyocr torch torchvision transformers
# !pip install gtts
# !apt-get install -y tesseract-ocr tesseract-ocr-fra

# ============================================
# IMPORTS
# ============================================
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch
from pathlib import Path
from collections import Counter
import easyocr
from transformers import BlipProcessor, BlipForConditionalGeneration
from torchvision import models
import time
import subprocess
import os
from gtts import gTTS
from IPython.display import Audio, display
from difflib import SequenceMatcher
import re

# ============================================
# GLOBAL CONFIGURATION
# ============================================
class ProjectConfig:
    """Centralized configuration for all modules"""

    # Models
    YOLO_MODEL = 'yolov8n.pt'
    CLASSIFIER_MODEL = 'resnet18'
    BLIP_MODEL = "Salesforce/blip-image-captioning-base"

    # Detection thresholds
    YOLO_CONF_THRESHOLD = 0.5
    OCR_CONF_THRESHOLD = 0.3

    # OCR
    OCR_LANGUAGES = ['fr', 'en']

    # TTS
    TTS_LANGUAGE = 'en'
    MIN_ALERT_INTERVAL = 5.0
    MAX_ALERTS_PER_SECOND = 2

    # Video processing intervals
    YOLO_INTERVAL = 5
    OCR_INTERVAL = 30
    BLIP_INTERVAL = 60
    CLASSIFIER_INTERVAL = 45

    # Critical obstacles
    CRITICAL_OBSTACLES = [
        'person', 'car', 'truck', 'bus', 'bicycle',
        'motorcycle', 'traffic light', 'stop sign', 'dog', 'cat'
    ]

# Create global instance
CONFIG = ProjectConfig()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# ============================================
# MODULE 1: YOLO - OBJECT DETECTION
# ============================================
class YOLOAnalyzer:
    """Object detection with YOLO"""

    def __init__(self, model, config):
        self.model = model
        self.config = config

    def analyze_frame(self, frame):
        """Analyze frame with YOLO"""
        results = self.model(frame, conf=self.config.YOLO_CONF_THRESHOLD, verbose=False)

        detections = []
        alertes = []

        h, w = frame.shape[:2]
        image_area = h * w

        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                xyxy = box.xyxy[0].cpu().numpy()
                class_name = self.model.names[cls]

                # Position
                center_x = (xyxy[0] + xyxy[2]) / 2
                if center_x < w / 3:
                    position = "à gauche"
                elif center_x < 2 * w / 3:
                    position = "devant"
                else:
                    position = "à droite"

                # Distance based on bbox size
                bbox_area = (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1])
                ratio = bbox_area / image_area
                if ratio > 0.3:
                    distance = "TRÈS PROCHE"
                elif ratio > 0.15:
                    distance = "Proche"
                else:
                    distance = "Loin"

                detection = {
                    'classe': class_name,
                    'confiance': conf,
                    'bbox': xyxy,
                    'distance': distance,
                    'position': position
                }
                detections.append(detection)

                # Alert for critical obstacles
                if class_name in self.config.CRITICAL_OBSTACLES:
                    alertes.append(f"{class_name} {position}, {distance}")

        return detections, alertes, results[0].plot()

# ============================================
# MODULE 2: OCR - TEXT READING
# ============================================
class OCRAnalyzer:
    """Text reading with EasyOCR"""

    def __init__(self, reader, config):
        self.reader = reader
        self.config = config

    def analyze_frame(self, frame):
        """Analyze frame with OCR"""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.reader.readtext(frame_rgb)

        textes = []
        for (bbox, text, conf) in results:
            if conf >= self.config.OCR_CONF_THRESHOLD:
                textes.append({
                    'texte': text.strip(),
                    'confiance': conf
                })

        return textes

# ============================================
# MODULE 3: BLIP - SCENE DESCRIPTION
# ============================================
class BLIPAnalyzer:
    """Scene description with BLIP"""

    def __init__(self, processor, model, device):
        self.processor = processor
        self.model = model
        self.device = device

    def analyze_frame(self, frame):
        """Generate frame description"""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_pil = Image.fromarray(frame_rgb)

        inputs = self.processor(image_pil, return_tensors="pt").to(self.device)
        outputs = self.model.generate(**inputs, max_length=50)
        description = self.processor.decode(outputs[0], skip_special_tokens=True)

        return description

# ============================================
# MODULE 4: CLASSIFIER
# ============================================
class ClassifierAnalyzer:
    """Object classification with ResNet"""

    def __init__(self, model, preprocess, labels, device):
        self.model = model
        self.preprocess = preprocess
        self.labels = labels
        self.device = device

    def analyze_frame(self, frame, top_k=3):
        """Classify frame content"""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_pil = Image.fromarray(frame_rgb)

        input_tensor = self.preprocess(image_pil).unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

        top_probs, top_idxs = probabilities.topk(top_k)

        results = []
        for prob, idx in zip(top_probs, top_idxs):
            results.append({
                'label': self.labels[int(idx)],
                'probabilite': float(prob)
            })

        return results

# ============================================
# MODEL LOADER
# ============================================
class ModelLoader:
    """Load all models once"""

    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f" Device: {self.device}")

        # YOLO
        self.yolo = YOLO(CONFIG.YOLO_MODEL)

        # OCR
        self.ocr_reader = easyocr.Reader(CONFIG.OCR_LANGUAGES, gpu=(self.device=='cuda'))

        # BLIP
        self.blip_processor = BlipProcessor.from_pretrained(CONFIG.BLIP_MODEL)
        self.blip_model = BlipForConditionalGeneration.from_pretrained(
            CONFIG.BLIP_MODEL
        ).to(self.device)

        # Classifier
        weights = models.ResNet18_Weights.DEFAULT
        self.classifier = models.resnet18(weights=weights)
        self.classifier.eval()
        self.classifier.to(self.device)
        self.classifier_preprocess = weights.transforms()
        self.classifier_labels = weights.meta["categories"]

# ============================================
# VIDEO PIPELINE
# ============================================
class VideoPipeline:
    """Complete video processing pipeline"""

    def __init__(self, config=None):
        self.config = config or CONFIG

        # Load models
        self.models = ModelLoader()

        # Initialize analyzers
        self.yolo_analyzer = YOLOAnalyzer(self.models.yolo, self.config)
        self.ocr_analyzer = OCRAnalyzer(self.models.ocr_reader, self.config)
        self.blip_analyzer = BLIPAnalyzer(
            self.models.blip_processor,
            self.models.blip_model,
            self.models.device
        )
        self.classifier_analyzer = ClassifierAnalyzer(
            self.models.classifier,
            self.models.classifier_preprocess,
            self.models.classifier_labels,
            self.models.device
        )

        # Stats
        self.stats = {
            'frames_totales': 0,
            'detections_yolo': [],
            'textes_ocr': [],
            'descriptions_blip': [],
            'classifications': []
        }

    def process_video(self, video_path, output_path=None, max_frames=None):
        """Process complete video"""
        print(f"\n TRAITEMENT VIDÉO: {video_path}")

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f" Error: cannot open {video_path}")
            return None

        # Video properties
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f" FPS: {fps} | Résolution: {width}x{height}")
        print(f" Frames totales: {total_frames}")
        print(f"  Durée: {total_frames/fps:.1f}s\n")

        # Writer
        out = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        frame_count = 0
        start_time = time.time()
        last_yolo_result = None
        current_annotations = None

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if max_frames and frame_count >= max_frames:
                break

            timestamp = frame_count / fps
            annotated_frame = frame.copy()

            # Module 1: YOLO
            if frame_count % self.config.YOLO_INTERVAL == 0:
                detections, alertes, yolo_annotated = self.yolo_analyzer.analyze_frame(frame)
                last_yolo_result = (detections, alertes)
                current_annotations = yolo_annotated

                if detections:
                    self.stats['detections_yolo'].extend(detections)
                    print(f" {timestamp:.1f}s |  YOLO: {len(detections)} objets")

                    if alertes:
                        for alerte in alertes[:3]:
                            print(f"           {alerte}")

            # Module 2: OCR
            if frame_count % self.config.OCR_INTERVAL == 0:
                textes = self.ocr_analyzer.analyze_frame(frame)
                if textes:
                    self.stats['textes_ocr'].extend(textes)
                    print(f" {timestamp:.1f}s |  OCR: {len(textes)} textes")
                    for t in textes[:2]:
                        print(f"         \"{t['texte']}\" ({t['confiance']:.2f})")

            # Module 3: BLIP
            if frame_count % self.config.BLIP_INTERVAL == 0:
                description = self.blip_analyzer.analyze_frame(frame)
                self.stats['descriptions_blip'].append(description)
                print(f" {timestamp:.1f}s |   BLIP: {description}")

            # Module 4: Classification
            if frame_count % self.config.CLASSIFIER_INTERVAL == 0:
                classifications = self.classifier_analyzer.analyze_frame(frame, top_k=2)
                self.stats['classifications'].extend(classifications)
                top = classifications[0]
                print(f" {timestamp:.1f}s |   Classe: {top['label']} ({top['probabilite']:.2f})")

            # Use YOLO annotations
            if current_annotations is not None:
                annotated_frame = current_annotations

            if out:
                out.write(annotated_frame)

            frame_count += 1

            if frame_count % 100 == 0:
                elapsed = time.time() - start_time
                fps_processing = frame_count / elapsed
                print(f"       Progression: {frame_count}/{total_frames} frames ({fps_processing:.1f} fps)")

        cap.release()
        if out:
            out.release()

        elapsed_total = time.time() - start_time

        print(f"\n TRAITEMENT TERMINÉ")
        print(f"  Temps: {elapsed_total:.1f}s")
        print(f" Vitesse: {frame_count/elapsed_total:.1f} fps")
        if output_path:
            print(f" Vidéo sauvegardée: {output_path}")

        return self.generate_summary()

    def generate_summary(self):
        """Generate analysis summary"""

        if self.stats['detections_yolo']:
            objets = [d['classe'] for d in self.stats['detections_yolo']]
            compteur = Counter(objets)
            print(f"\n YOLO - {len(self.stats['detections_yolo'])} détections totales")
            print("Top 5 objets:")
            for obj, count in compteur.most_common(5):
                print(f"   • {obj}: {count}x")

        if self.stats['textes_ocr']:
            print(f"\n OCR - {len(self.stats['textes_ocr'])} textes détectés")
            textes_uniques = set(t['texte'] for t in self.stats['textes_ocr'])
            print(f"Textes uniques: {len(textes_uniques)}")
            for texte in list(textes_uniques)[:5]:
                print(f"   • \"{texte}\"")

        if self.stats['descriptions_blip']:
            print(f"\n  BLIP - {len(self.stats['descriptions_blip'])} descriptions")
            print("Exemples:")
            for desc in self.stats['descriptions_blip'][:3]:
                print(f"   • {desc}")

        if self.stats['classifications']:
            labels = [c['label'] for c in self.stats['classifications']]
            compteur = Counter(labels)
            print(f"\n  Classification - Top 5 catégories:")
            for label, count in compteur.most_common(5):
                print(f"   • {label}: {count}x")

        return self.stats

# ============================================
# TTS SYSTEM
# ============================================
class CleanTTS:
    """Professional TTS with filtering"""

    def __init__(self, lang='en'):
        self.audio_counter = 0
        self.last_alerts = {}
        self.lang = lang
        self.spoken_texts = []
        print(f"🔊 TTS initialized ({lang.upper()})")

    def speak(self, text, force=False):
        """Speak with duplicate detection"""
        if not text or text.strip() == "":
            return False

        current_time = time.time()
        if not force and text in self.last_alerts:
            if current_time - self.last_alerts[text] < 5:
                return False

        self.last_alerts[text] = current_time
        self.spoken_texts.append(text)

        print(f"🔊 {text}")

        try:
            self.audio_counter += 1
            filename = f'audio_{self.audio_counter}.mp3'

            tts = gTTS(text=text, lang=self.lang, slow=False)
            tts.save(filename)

            display(Audio(filename, autoplay=True))

            words = len(text.split())
            duration = max(words * 0.4, 1.5)
            time.sleep(duration)

            return True

        except Exception as e:
            print(f" TTS Error: {e}")
            return False

class SmartTextFilter:
    """Text filtering utilities"""

    @staticmethod
    def similarity_ratio(text1, text2):
        return SequenceMatcher(None, text1.lower(), text2.lower()).ratio()

    @staticmethod
    def deduplicate_texts(texts, min_similarity=0.85):
        if not texts:
            return []

        sorted_texts = sorted(texts, key=lambda x: x['confiance'], reverse=True)

        unique = []
        for text_obj in sorted_texts:
            text = text_obj['texte']

            is_duplicate = False
            for kept in unique:
                if SmartTextFilter.similarity_ratio(text, kept['texte']) >= min_similarity:
                    is_duplicate = True
                    break

            if not is_duplicate:
                unique.append(text_obj)

        return unique

    @staticmethod
    def is_street_name(text):
        street_keywords = [
            'AVENUE', 'STREET', 'ROAD', 'BOULEVARD', 'RUE', 'PLACE',
            'PALISSY', 'BERNARD', 'WAY', 'LANE', 'DRIVE'
        ]
        text_upper = text.upper()
        return any(kw in text_upper for kw in street_keywords)

class AlertFormatter:
    """Format alerts in natural language"""

    @staticmethod
    def format_yolo_alert(detection):
        classe = detection['classe']
        position = detection['position']
        distance = detection['distance']

        pos_map = {
            'à gauche': 'on your left',
            'devant': 'ahead of you',
            'à droite': 'on your right'
        }
        pos_en = pos_map.get(position, position)

        if distance == "TRÈS PROCHE":
            return f"WARNING! {classe} very close {pos_en}", 'critical'
        elif distance == "Proche":
            return f"{classe} {pos_en}", 'warning'
        else:
            return None, None

    @staticmethod
    def format_ocr_alert(text_obj):
        text = text_obj['texte']
        conf = text_obj['confiance']

        if conf < 0.5 or len(text) < 3:
            return None, None

        if SmartTextFilter.is_street_name(text):
            return f"Street sign {text}", 'info'

        if len(text) > 5:
            return f"Sign {text}", 'info'

        return None, None

class VideoAssistant:
    """Video assistant with smart TTS"""

    def __init__(self, lang='en'):
        self.tts = CleanTTS(lang=lang)
        self.filter = SmartTextFilter()
        self.formatter = AlertFormatter()

    def process_results(self, stats):
        """Process results and generate alerts"""

        alerts_spoken = 0

        # Critical alerts
        print("\n Critical Alerts:")
        critical_count = 0
        if stats.get('detections_yolo'):
            seen = set()
            for det in stats['detections_yolo']:
                if det['distance'] == 'TRÈS PROCHE':
                    alert, priority = self.formatter.format_yolo_alert(det)
                    if alert and alert not in seen:
                        if self.tts.speak(alert):
                            critical_count += 1
                            alerts_spoken += 1
                        seen.add(alert)

        if critical_count == 0:
            print("   None")

        # Warning alerts
        print("\n Warning Alerts:")
        warning_count = 0
        if stats.get('detections_yolo'):
            seen = set()
            for det in stats['detections_yolo'][:15]:
                if det['distance'] == 'Proche':
                    alert, priority = self.formatter.format_yolo_alert(det)
                    if alert and alert not in seen:
                        if self.tts.speak(alert):
                            warning_count += 1
                            alerts_spoken += 1
                        seen.add(alert)
                        if warning_count >= 3:
                            break

        if warning_count == 0:
            print("   None")

        # Information
        print("\n  Information:")
        info_count = 0
        if stats.get('textes_ocr'):
            unique_texts = self.filter.deduplicate_texts(
                stats['textes_ocr'],
                min_similarity=0.85
            )

            for text_obj in unique_texts[:4]:
                alert, priority = self.formatter.format_ocr_alert(text_obj)
                if alert:
                    if self.tts.speak(alert):
                        info_count += 1
                        alerts_spoken += 1

        if info_count == 0:
            print("   None")

        # Environment
        print("\n Environment:")
        if stats.get('descriptions_blip') and len(stats['descriptions_blip']) > 0:
            desc = stats['descriptions_blip'][0]
            self.tts.speak(f"You are in {desc}")
            alerts_spoken += 1
        else:
            print("   No description available")

        print(f"\n Total alerts spoken: {alerts_spoken}")

        return alerts_spoken

def process_video_for_blind_assistance(
    video_path,
    max_frames=None,
    output_video_path=None,
    language='en',
    config=None
):
    """
    Main function to process video with blind assistance

    Args:
        video_path: Path to input video
        max_frames: Max frames to process (None = all)
        output_video_path: Path to save annotated video
        language: 'en' or 'fr'
        config: ProjectConfig instance

    Returns:
        stats, assistant
    """
    config = config or CONFIG


    # Analyze video
    pipeline = VideoPipeline(config)
    stats = pipeline.process_video(
        video_path,
        output_path=output_video_path,
        max_frames=max_frames
    )

    # Generate voice alerts
    assistant = VideoAssistant(lang=language)
    assistant.process_results(stats)



    return stats, assistant

# ============================================
# DEMO WITH AUDIO
# ============================================
def create_final_demo_with_audio(video_path, output_path='FINAL_DEMO_WITH_AUDIO.mp4'):
    """Create final demo with audio alerts"""


    # Process video
    temp_video = 'temp_annotated_video.mp4'
    pipeline = VideoPipeline(config=CONFIG)
    stats = pipeline.process_video(video_path, output_path=temp_video)

    print(f"\n Statistics:")
    print(f"   Detections: {len(stats['detections_yolo'])}")
    print(f"   OCR texts: {len(stats['textes_ocr'])}")
    print(f"   Descriptions: {len(stats['descriptions_blip'])}")

    # Generate audio alerts
    print("\n Generating audio alerts.")
    audio_alerts = []

    # Car alert
    car_found = False
    for det in stats['detections_yolo']:
        if det['classe'] in ['car', 'truck', 'bus'] and not car_found:
            pos_en = {
                'à gauche': 'on the left',
                'devant': 'ahead',
                'à droite': 'on the right'
            }.get(det['position'], det['position'])

            dist_en = {
                'TRÈS PROCHE': 'very close',
                'Proche': 'close',
                'Loin': 'far'
            }.get(det['distance'], det['distance'])

            alert_text = f"Warning! {det['classe']} detected {dist_en}, {pos_en}"
            audio_alerts.append(alert_text)
            car_found = True
            break

    # Street alert
    street_found = False
    if stats['textes_ocr']:
        for text_obj in stats['textes_ocr']:
            text = text_obj['texte']
            street_keywords = ['AVENUE', 'BERNARD', 'PALISSY', 'STREET', 'RUE', 'ROAD']

            if any(kw in text.upper() for kw in street_keywords) and not street_found:
                alert_text = f"Street sign detected, {text}"
                audio_alerts.append(alert_text)
                street_found = True
                break

    # Environment
    if stats['descriptions_blip']:
        desc = stats['descriptions_blip'][0]
        alert_text = f"You are in {desc}"
        audio_alerts.append(alert_text)

    # Generate audio files
    audio_files = []
    for i, alert in enumerate(audio_alerts):
        filename = f'alert_{i}.mp3'
        try:
            tts = gTTS(text=alert, lang='en', slow=False)
            tts.save(filename)
            audio_files.append(filename)
            print(f"   {filename}: \"{alert}\"")
        except Exception as e:
            print(f"   Error: {e}")

    # Merge video + audio
    print("\nMerging video with audio.")

    if audio_files:
        audio_file = audio_files[0]

        cmd = [
            'ffmpeg', '-y',
            '-i', temp_video,
            '-i', audio_file,
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-map', '0:v:0',
            '-map', '1:a:0',
            '-shortest',
            output_path
        ]

        try:
            subprocess.run(cmd, capture_output=True, timeout=120)
            print(f"    Successfully merged!")
        except:
            print(f"     Merge warning, copying video")
            import shutil
            shutil.copy(temp_video, output_path)
    else:
        print("   No audio to merge")
        import shutil
        shutil.copy(temp_video, output_path)

    print(f"\n Final video: {output_path}")

    # Cleanup
    try:
        os.remove(temp_video)
        for audio_file in audio_files:
            os.remove(audio_file)
    except:
        pass

    return output_path, stats


if __name__ == "__main__":

    stats, assistant = process_video_for_blind_assistance(
        video_path='/content/IMG_1938.MOV',
        max_frames=300,
        language='en'
    )


    final_video, stats = create_final_demo_with_audio(
        video_path='/content/IMG_1938.MOV',
        output_path='FINAL_DEMO_WITH_AUDIO.mp4'
    )

PyTorch version: 2.9.0+cu126
CUDA available: True
 Device: cuda

 TRAITEMENT VIDÉO: /content/IMG_1938.MOV
 FPS: 29 | Résolution: 1080x1920
 Frames totales: 396
  Durée: 13.7s

 0.0s |  YOLO: 2 objets
           car à gauche, Proche
 0.0s |  OCR: 2 textes
         "3780" (0.53)
         "AX-558 KVE" (0.80)
 0.0s |   BLIP: a car parked on the side of a road
 0.0s |   Classe: minivan (0.58)
 0.2s |  YOLO: 2 objets
           car à gauche, Proche
 0.3s |  YOLO: 1 objets
           car devant, Proche
 0.5s |  YOLO: 2 objets
           car devant, Proche
 0.7s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 0.9s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 1.0s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.0s |  OCR: 4 textes
         "cdi" (0.31)
         "5180" (0.36)
 1.2s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 1.4s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.6s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.6s |   Classe: miniv


 Warning Alerts:
🔊 car on your left


🔊 car ahead of you



  Information:
🔊 Sign AX558 KV



 Environment:
🔊 You are in a car parked on the side of a road



 Total alerts spoken: 5
 Device: cuda

 TRAITEMENT VIDÉO: /content/IMG_1938.MOV
 FPS: 29 | Résolution: 1080x1920
 Frames totales: 396
  Durée: 13.7s

 0.0s |  YOLO: 2 objets
           car à gauche, Proche
 0.0s |  OCR: 2 textes
         "3780" (0.52)
         "AX-558 KVE" (0.80)
 0.0s |   BLIP: a car parked on the side of a road
 0.0s |   Classe: minivan (0.58)
 0.2s |  YOLO: 2 objets
           car à gauche, Proche
 0.3s |  YOLO: 1 objets
           car devant, Proche
 0.5s |  YOLO: 2 objets
           car devant, Proche
 0.7s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 0.9s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 1.0s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.0s |  OCR: 4 textes
         "cdi" (0.31)
         "5180" (0.36)
 1.2s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 1.4s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.6s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.6s |   Classe: minivan (0.62)
 1.7s |  YOLO: 

In [110]:
stats, assistant = process_video_for_blind_assistance(
    video_path='/content/IMG_1938.MOV',
    max_frames=100,  # Test rapide sur 100 frames
    language='en'
)

 Device: cuda

 TRAITEMENT VIDÉO: /content/IMG_1938.MOV
 FPS: 29 | Résolution: 1080x1920
 Frames totales: 396
  Durée: 13.7s

 0.0s |  YOLO: 2 objets
           car à gauche, Proche
 0.0s |  OCR: 2 textes
         "3780" (0.52)
         "AX-558 KVE" (0.80)
 0.0s |   BLIP: a car parked on the side of a road
 0.0s |   Classe: minivan (0.58)
 0.2s |  YOLO: 2 objets
           car à gauche, Proche
 0.3s |  YOLO: 1 objets
           car devant, Proche
 0.5s |  YOLO: 2 objets
           car devant, Proche
 0.7s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 0.9s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 1.0s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.0s |  OCR: 4 textes
         "cdi" (0.31)
         "5180" (0.36)
 1.2s |  YOLO: 2 objets
           car devant, TRÈS PROCHE
 1.4s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.6s |  YOLO: 1 objets
           car devant, TRÈS PROCHE
 1.6s |   Classe: minivan (0.62)
 1.7s |  YOLO: 1 objets
           car d


 Warning Alerts:
🔊 car on your left


🔊 car ahead of you



  Information:
🔊 Sign AX558 KV



 Environment:
🔊 You are in a car parked on the side of a road



 Total alerts spoken: 5


LIMITATIONS


1. Distance Estimation:
   - Currently uses bounding box size as proxy for distance
   - NOT metric (meters) - only relative (close/far)
   - Accuracy depends on object type and camera calibration
   - Future: Implement depth estimation with stereo camera

2. Performance:
   - Processing 4 heavy models per frame is computationally expensive
   - Current speed: ~5-10 FPS on GPU, ~1-2 FPS on CPU
   - Not true real-time without optimization (TensorRT, quantization)

3. OCR Accuracy:
   - Works best with clear, large text
   - Performance degrades with: poor lighting, motion blur, small text
   - French + English mixed text can confuse the model

4. Internet Dependency:
   - gTTS requires internet connection
   - Alternative: Use pyttsx3 for offline (lower quality voice)

5. Alert Spam:
   - System filters duplicate alerts (5s cooldown)
   - May miss important changes if they happen too quickly